# Trabajo Práctico N°2 - Sistemas Inteligentes con RAG o Agentes

## Asistente de Mantenimiento: consulta de manuales técnicos de máquinas industriales

**Integrantes:** _completar_

Herramientas: LangChain, Google Gemini (`gemini-3.1-flash-lite` como LLM y `gemini-embedding-001` para los embeddings), Chroma y BM25.

## 1. Caso de negocio

En una planta industrial, cuando una máquina marca una alarma o hay que hacerle mantenimiento, el técnico tiene que buscar la información en los manuales del fabricante: qué significa el código de alarma, qué procedimiento seguir, qué lubricante lleva, qué torque aplicar o cada cuántas horas se hace cada tarea. Son manuales largos, de distintos fabricantes, y mientras se busca la máquina sigue parada.

El Asistente de Mantenimiento es un sistema experto para hacer esas consultas en lenguaje natural (por ejemplo, "el torno marca AL-205, ¿qué hago?"). Para cada pregunta:

1. decide en qué manual buscar, según la máquina que se nombra en la pregunta o en la conversación;
2. recupera los fragmentos más relevantes de ese manual (RAG con búsqueda híbrida: embeddings + BM25);
3. le pasa esos fragmentos al LLM, que responde citando la fuente, o avisa que no encontró la información;
4. aprende de las conversaciones: cuando el técnico marca una respuesta como correcta (👍), la recuerda y la usa en las consultas parecidas que vengan después.

Las conversaciones no se guardan. Lo único que persiste es lo aprendido: cada pregunta validada con su respuesta y los fragmentos del manual que citó.

Elegimos RAG y no un agente porque el problema es recuperar información de documentos: el modelo no necesita usar herramientas, y así cada pregunta se resuelve con una sola llamada al LLM.

## 2. Configuración

Hace falta una API key de Gemini en el archivo `.env` (ver `.env.example`). Las dependencias se instalan con `pip install -r requirements.txt`.

Parámetros principales:
- temperatura 0.1, porque en respuestas técnicas importa más la precisión que la variedad;
- chunks de 900 caracteres con 150 de solapamiento;
- se le envían al modelo los 4 fragmentos más relevantes (`TOP_K`);
- una consulta validada se reutiliza si su similitud con la nueva es de 0,88 o más (`UMBRAL_MEMORIA`, sección 11.1), y se usan como máximo 2 por pregunta (`MAX_RECUERDOS`).

El código de este notebook es el mismo que usa el server de la aplicación web (carpeta `asistente/`, ver sección 12).

In [1]:
import logging
import os
import re
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 100)

In [2]:
BASE_DIR = Path.cwd()  # carpeta server/, donde está este notebook
MANUALES_DIR = BASE_DIR / "data" / "manuales"
CHROMA_DIR = BASE_DIR / "chroma_db"
load_dotenv(BASE_DIR / ".env")
logging.getLogger("google_genai.models").setLevel(logging.ERROR)  # oculta un aviso interno del SDK de Gemini

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "models/gemini-embedding-001")
TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.1"))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "900"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "150"))
TOP_K = int(os.getenv("TOP_K", "4"))  # fragmentos que se envían al modelo
COLLECTION_NAME = "manuales"
# Aprendizaje: una consulta validada se reutiliza si su similitud con la nueva supera el umbral
# (0.88: las paráfrasis medidas dan 0.90 o más, y dos preguntas distintas sobre el aceite del compresor, 0.867)
UMBRAL_MEMORIA = float(os.getenv("UMBRAL_MEMORIA", "0.88"))
MAX_RECUERDOS = int(os.getenv("MAX_RECUERDOS", "2"))  # consultas validadas que se usan por pregunta


def get_llm():
    return ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=TEMPERATURE)


def get_embeddings(task_type=None):
    """task_type=None usa RETRIEVAL_DOCUMENT para los chunks y RETRIEVAL_QUERY para las consultas."""
    return GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, task_type=task_type)


print("API key cargada:", bool(os.getenv("GOOGLE_API_KEY")))
print("Modelo:", GEMINI_MODEL, "| Embeddings:", EMBEDDING_MODEL)

API key cargada: True
Modelo: gemini-3.1-flash-lite | Embeddings: models/gemini-embedding-001


## 3. Base de conocimiento: los manuales

| Modelo | Equipo | Alias (cómo lo nombran los usuarios) |
|---|---|---|
| CT-75 | Compresor de tornillo AERON | compresor, aeron |
| TC-420 | Torno CNC TORVEX | torno, torno cnc, torvex |
| IP-250 | Inyectora de plástico PLASTIMAQ | inyectora, plastimaq |
| CB-1200 | Cinta transportadora TRANSBELT | cinta, cinta transportadora, transbelt |
| PRO-SEG-001 | Procedimiento de bloqueo y etiquetado (LOTO) | loto, bloqueo, etiquetado, candado |

Los equipos son ficticios a propósito. Si usáramos manuales de máquinas conocidas, el modelo podría responder con lo que aprendió en su entrenamiento y no sabríamos si la respuesta vino de los documentos. Con datos inventados (alarmas, torques e intervalos que no existen en internet), cada respuesta correcta tiene que salir del RAG.

Cada manual empieza con un encabezado con el modelo, el equipo y los alias, que después se usan para detectar de qué máquina habla una pregunta.

In [3]:
for ruta in sorted(MANUALES_DIR.glob("*.md")):
    print(f"{ruta.name:48s} {ruta.stat().st_size / 1024:5.1f} KB")

print()
print((MANUALES_DIR / "CT-75_compresor_tornillo_aeron.md").read_text(encoding="utf-8")[:900])

CB-1200_cinta_transportadora_transbelt.md          4.7 KB
CT-75_compresor_tornillo_aeron.md                  6.7 KB
IP-250_inyectora_plastimaq.md                      6.2 KB
PRO-SEG-001_bloqueo_etiquetado_LOTO.md             3.5 KB
TC-420_torno_cnc_torvex.md                         5.9 KB

---
modelo: CT-75
equipo: Compresor de tornillo AERON CT-75
fabricante: Aeron Industrial S.A. (ficticio)
alias: compresor, compresor de tornillo, aeron
revision: Rev. 3 - 2025
---

# Manual de operación y mantenimiento - Compresor de tornillo AERON CT-75

> Documento ficticio creado con fines académicos. Los valores no corresponden a ningún equipo comercial real.

## 1. Descripción general

El AERON CT-75 es un compresor de aire rotativo de tornillo, lubricado por inyección de aceite, de una etapa, refrigerado por aire. Está diseñado para servicio industrial continuo (24/7) en plantas metalúrgicas, alimenticias y de plásticos.

### 1.1 Datos técnicos

| Parámetro | Valor |
|---|---|
| Potencia del mo

## 4. Carga y división en fragmentos (chunking)

No se le pueden pasar al modelo todos los manuales en cada pregunta, así que se dividen en fragmentos (chunks) y solo se envían los relevantes. La división se hace en dos pasos:

1. por secciones del manual (`MarkdownHeaderTextSplitter`), para que un chunk no mezcle, por ejemplo, las alarmas con la lubricación;
2. por tamaño (`RecursiveCharacterTextSplitter`), cortando preferentemente entre párrafos o entre líneas.

Además, a cada chunk se le agrega al principio el equipo y la sección de donde sale.

In [4]:
def leer_front_matter(texto):
    """Separa el encabezado '--- clave: valor ---' del cuerpo del manual."""
    m = re.match(r"^---\s*\n(.*?)\n---\s*\n", texto, re.DOTALL)
    if not m:
        return {}, texto
    meta = {}
    for linea in m.group(1).splitlines():
        if ":" in linea:
            clave, valor = linea.split(":", 1)
            meta[clave.strip()] = valor.strip()
    return meta, texto[m.end():]


def cargar_manuales(directorio=MANUALES_DIR):
    docs = []
    for ruta in sorted(Path(directorio).glob("*.md")):
        meta, cuerpo = leer_front_matter(ruta.read_text(encoding="utf-8"))
        # la aclaración "Documento ficticio" no aporta nada técnico
        cuerpo = "\n".join(l for l in cuerpo.splitlines() if not l.startswith("> Documento ficticio"))
        docs.append(Document(page_content=cuerpo, metadata={
            "archivo": ruta.name,
            "modelo": meta.get("modelo", ruta.stem),
            "equipo": meta.get("equipo", ruta.stem),
            "alias": meta.get("alias", ""),
        }))
    return docs

In [5]:
splitter_secciones = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])
splitter_tamano = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
                                                 separators=["\n\n", "\n", ". ", " ", ""])


def dividir(docs):
    """Divide cada manual primero por secciones y después por tamaño.

    A cada chunk se le agrega al principio el equipo y la sección de donde sale,
    para que no quede un fragmento suelto que no dice de qué máquina habla.
    """
    chunks = []
    for doc in docs:
        secciones = splitter_secciones.split_text(doc.page_content)
        for s in secciones:
            titulos = [s.metadata[h] for h in ("h2", "h3") if h in s.metadata]
            s.metadata = {**doc.metadata, "seccion": " > ".join(titulos) or "General"}
        for n, chunk in enumerate(splitter_tamano.split_documents(secciones)):
            m = chunk.metadata
            chunk.page_content = f"[{m['equipo']} ({m['modelo']}) | {m['seccion']}]\n{chunk.page_content.strip()}"
            chunk.id = f"{m['archivo']}::{n:03d}"
            chunks.append(chunk)
    return chunks

In [6]:
docs = cargar_manuales()
chunks = dividir(docs)
print(f"Manuales: {len(docs)} | Chunks: {len(chunks)}")

largos = pd.DataFrame({"modelo": [c.metadata["modelo"] for c in chunks],
                       "caracteres": [len(c.page_content) for c in chunks]})
largos.groupby("modelo")["caracteres"].agg(["count", "mean", "min", "max"]).round(0)

Manuales: 5 | Chunks: 54


,count,mean,min,max
modelo,,,,
CB-1200,10,493.0,275,802
CT-75,13,550.0,301,878
IP-250,11,596.0,274,979
PRO-SEG-001,7,527.0,340,939
TC-420,13,493.0,225,873


In [7]:
# Un chunk de ejemplo: la primera línea es el encabezado con el equipo y la sección
ejemplo = next(c for c in chunks if "A01" in c.page_content and "Detalle" in c.metadata["seccion"])
print(ejemplo.id, "|", ejemplo.metadata["seccion"])
print()
print(ejemplo.page_content)

CT-75_compresor_tornillo_aeron.md::011 | 5. Alarmas y diagnóstico > 5.1 Detalle de la alarma A01

[Compresor de tornillo AERON CT-75 (CT-75) | 5. Alarmas y diagnóstico > 5.1 Detalle de la alarma A01]
La advertencia de temperatura se activa a 105 °C y la parada por alarma A01 ocurre a 110 °C de temperatura de descarga. No rearmar el equipo más de dos veces consecutivas sin identificar la causa: el funcionamiento repetido a alta temperatura degrada el aceite y puede dañar los rodamientos del elemento. Si la temperatura ambiente de la sala supera los 40 °C, instalar ductos de extracción de aire caliente.


Para ver si el encabezado sirve, lo comparamos con cortar el texto solo por tamaño: contamos cuántos chunks no dicen en su texto de qué máquina son.

In [8]:
ingenuo = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks_ingenuos = ingenuo.split_documents(docs)


def menciona_equipo(chunk):
    marca = chunk.metadata["equipo"].split()[-2]  # por ejemplo "AERON"
    texto = chunk.page_content.upper()
    return chunk.metadata["modelo"].upper() in texto or marca.upper() in texto


for nombre, lista in [("Solo por tamaño", chunks_ingenuos), ("Secciones + encabezado", chunks)]:
    sin_equipo = sum(not menciona_equipo(c) for c in lista)
    print(f"{nombre:23s}: {sin_equipo} de {len(lista)} chunks no dicen de qué equipo son")

Solo por tamaño        : 31 de 42 chunks no dicen de qué equipo son
Secciones + encabezado : 0 de 54 chunks no dicen de qué equipo son


Cortando solo por tamaño, 31 de los 42 chunks (74 %) no dicen en ningún lado de qué máquina son. Con la división por secciones y el encabezado, ninguno. Un fragmento como "no rearmar el equipo más de dos veces consecutivas" sin encabezado podría salir en una pregunta sobre cualquier otra máquina.

## 5. Embeddings

Un embedding representa un texto como un vector de números, de forma que textos con significado parecido quedan cerca aunque usen palabras distintas. La cercanía se mide con la similitud coseno:

$$\text{sim}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

Probamos con cinco frases: las dos primeras describen el mismo problema con otras palabras, y las dos últimas son códigos de alarma distintos (AL-205 existe en el manual del torno, AL-250 no).

In [9]:
embeddings = get_embeddings()
frases = [
    "El torno tiene presión hidráulica baja",
    "La máquina pierde fuerza para sujetar la pieza por falta de aceite",
    "Cambiar el aceite del compresor cada 4.000 horas",
    "Alarma AL-205",
    "Alarma AL-250",
]
vectores = np.array(embeddings.embed_documents(frases))
print("Dimensión de cada vector:", vectores.shape[1])

normalizados = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
pd.DataFrame(normalizados @ normalizados.T, index=frases, columns=["F1", "F2", "F3", "F4", "F5"]).round(3)

Dimensión de cada vector: 3072


,F1,F2,F3,F4,F5
El torno tiene presión hidráulica baja,1.000,0.860,0.755,0.722,0.729
La máquina pierde fuerza para sujetar la pieza por falta de aceite,0.860,1.000,0.782,0.695,0.694
Cambiar el aceite del compresor cada 4.000 horas,0.755,0.782,1.000,0.706,0.710
Alarma AL-205,0.722,0.695,0.706,1.000,0.940
Alarma AL-250,0.729,0.694,0.710,0.940,1.000


Las frases 1 y 2 tienen una similitud de 0,86 aunque casi no comparten palabras: los embeddings reconocen que describen el mismo problema. En cambio, "Alarma AL-205" y "Alarma AL-250" dan 0,94, casi lo mismo, aunque son alarmas distintas. Los embeddings capturan de qué se habla pero no distinguen bien un código de otro parecido, y por eso sumamos BM25 en la sección 7.

## 6. Índice vectorial (Chroma)

Los embeddings de los chunks se guardan en Chroma, en disco (carpeta `chroma_db/`), con distancia coseno. Si el índice ya existe se reutiliza, para no volver a calcular los embeddings cada vez.

In [10]:
def obtener_vectorstore():
    return Chroma(collection_name=COLLECTION_NAME, embedding_function=get_embeddings(),
                  persist_directory=str(CHROMA_DIR), collection_metadata={"hnsw:space": "cosine"})


def construir_indice(vs):
    """Borra el índice y vuelve a indexar todos los manuales. Devuelve la cantidad de chunks."""
    vs.reset_collection()
    chunks = dividir(cargar_manuales())
    vs.add_documents(chunks, ids=[c.id for c in chunks])
    return len(chunks)

In [11]:
vectorstore = obtener_vectorstore()
if not vectorstore.get(include=[])["ids"]:
    print("Índice vacío, indexando:", construir_indice(vectorstore), "chunks")

metadatos = vectorstore.get(include=["metadatas"])["metadatas"]
pd.DataFrame(metadatos).groupby(["modelo", "equipo"]).size().rename("chunks").reset_index()

,modelo,equipo,chunks
0,CB-1200,Cinta transportadora TRANSBELT CB-1200,10
1,CT-75,Compresor de tornillo AERON CT-75,13
2,IP-250,Inyectora de plástico PLASTIMAQ IP-250,11
3,PRO-SEG-001,Procedimiento general de planta - Bloqueo y etiquetado (LOTO),7
4,TC-420,Torno CNC TORVEX TC-420,13


## 7. Recuperación híbrida: embeddings + BM25

En los manuales aparecen dos tipos de consultas:

- **semánticas**, donde el problema se describe con otras palabras ("la banda resbala" cuando el manual dice "patina"). Acá funcionan bien los embeddings;
- **exactas**, con códigos de alarma o medidas (`AL-205`, `E-17`, `M16`). Los embeddings no distinguen bien estos códigos, pero BM25 sí, porque compara las palabras.

Por eso usamos las dos búsquedas y combinamos los rankings con *Reciprocal Rank Fusion* (RRF): cada chunk suma $\frac{1}{60 + \text{posición}}$ en cada lista donde aparece. RRF no necesita normalizar los puntajes, que en cada buscador están en escalas distintas.

Para BM25 hicimos un tokenizador propio: pasa a minúsculas, saca acentos y stopwords, y conserva los códigos como `al-205` (agregando también `al` y `205`).

In [12]:
RRF_K = 60  # constante usual de Reciprocal Rank Fusion
STOPWORDS = set("""
a al algo ante como con contra cual cuales cuando de del desde donde durante e el ella
ellos en entre era es esa ese eso esta este esto hay la las le les lo los mas me mi muy
no o para pero por que se sea ser si sin sobre su sus tambien te tiene un una uno unos
y ya hace hacer debo tengo cada cuanto cuantos cuanta cuantas qué cómo puede
""".split())


def sin_acentos(texto):
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))


def tokenizar(texto):
    """Tokenizador para BM25: minúsculas, sin acentos y sin stopwords.

    Conserva los códigos ("al-205") y además agrega sus partes ("al", "205"),
    y los números con punto de miles también se agregan sin él ("4.000" -> "4000").
    """
    tokens = []
    for tok in re.findall(r"[a-z0-9]+(?:[-.,][a-z0-9]+)*", sin_acentos(texto.lower())):
        if tok in STOPWORDS:
            continue
        tokens.append(tok)
        if "-" in tok:
            tokens.extend(p for p in tok.split("-") if p)
        if re.fullmatch(r"\d+([.,]\d+)+", tok):
            tokens.append(re.sub(r"[.,]", "", tok))
    return tokens

In [13]:
tokenizar("La alarma AL-205 indica presión < 35 bar; cambio de aceite cada 4.000 horas.")

['alarma',
 'al-205',
 'al',
 '205',
 'indica',
 'presion',
 '35',
 'bar',
 'cambio',
 'aceite',
 '4.000',
 '4000',
 'horas']

In [14]:
class RecuperadorHibrido:
    def __init__(self, vectorstore, candidatos=12):
        self.vs = vectorstore
        self.candidatos = candidatos  # resultados que aporta cada buscador antes de fusionar
        # BM25 se arma con los mismos chunks guardados en Chroma
        datos = vectorstore.get(include=["documents", "metadatas"])
        self.docs = [Document(page_content=t, metadata=m, id=i)
                     for i, t, m in zip(datos["ids"], datos["documents"], datos["metadatas"])]
        self.por_id = {d.id: d for d in self.docs}
        self.bm25 = BM25Okapi([tokenizar(d.page_content) for d in self.docs])
        self.patrones = self._patrones_por_manual()

    @property
    def modelos(self):
        return sorted({d.metadata["modelo"] for d in self.docs})

    # --- Detección del manual (sin usar el LLM) ---
    def _patrones_por_manual(self):
        """Para cada manual, expresiones regulares con su código de modelo y sus alias."""
        palabras = {}
        for d in self.docs:
            m = d.metadata
            palabras.setdefault(m["modelo"], {m["modelo"]}).update(
                a.strip() for a in m.get("alias", "").split(",") if a.strip())
        patrones = {}
        for modelo, claves in palabras.items():
            lista = []
            for clave in claves:
                clave = sin_acentos(clave.lower())
                for v in {clave, clave.replace("-", ""), clave.replace("-", " ")}:  # CT-75, CT75, CT 75
                    lista.append(re.compile(rf"\b{re.escape(v)}(es|s)?\b"))  # admite plural
            patrones[modelo] = lista
        return patrones

    def detectar_manual(self, texto):
        """Devuelve el manual que nombra el texto, o None si no nombra ninguno o nombra varios."""
        texto = sin_acentos(texto.lower())
        encontrados = [modelo for modelo, pats in self.patrones.items() if any(p.search(texto) for p in pats)]
        return encontrados[0] if len(encontrados) == 1 else None

    # --- Búsqueda ---
    def embeber(self, texto):
        """Embedding de la consulta: se calcula una vez y se usa para los manuales y para la memoria."""
        return self.vs.embeddings.embed_query(texto)

    def buscar_vectorial(self, consulta, k, modelo=None, vector=None):
        filtro = {"modelo": modelo} if modelo else None
        vector = vector if vector is not None else self.embeber(consulta)
        resultados = self.vs.similarity_search_by_vector_with_relevance_scores(vector, k=k, filter=filtro)
        return [(d, round(1 - dist, 4)) for d, dist in resultados]  # distancia coseno -> similitud

    def buscar_bm25(self, consulta, k, modelo=None):
        puntajes = self.bm25.get_scores(tokenizar(consulta))
        orden = sorted(range(len(self.docs)), key=lambda i: puntajes[i], reverse=True)
        resultado = []
        for i in orden:
            if puntajes[i] <= 0 or len(resultado) == k:
                break
            if modelo and self.docs[i].metadata["modelo"] != modelo:
                continue
            resultado.append((self.docs[i], round(float(puntajes[i]), 4)))
        return resultado

    def buscar_aprendidos(self, recuerdos, modelo=None):
        """Fragmentos que citaron las respuestas validadas parecidas, puntuados con la similitud de cada recuerdo."""
        resultado, vistos = [], set()
        for r in recuerdos:
            for fid in r["fragmentos"]:
                doc = self.por_id.get(fid)  # si el manual se reindexó, el fragmento puede no existir
                if doc and fid not in vistos and (not modelo or doc.metadata["modelo"] == modelo):
                    vistos.add(fid)
                    resultado.append((doc, r["similitud"]))
        return resultado

    def buscar(self, consulta, k=5, modelo=None, modo="hibrido", vector=None, recuerdos=()):
        """Devuelve los k chunks más relevantes. modo: "hibrido", "vectorial" o "bm25".

        recuerdos: consultas validadas parecidas (memoria); sus fragmentos entran como una lista más de RRF.
        """
        vect = self.buscar_vectorial(consulta, self.candidatos, modelo, vector) if modo != "bm25" else []
        lex = self.buscar_bm25(consulta, self.candidatos, modelo) if modo != "vectorial" else []
        mem = self.buscar_aprendidos(recuerdos, modelo)

        # RRF: cada lista suma 1 / (60 + posición) a los chunks que encontró
        fusion = {}
        for nombre, lista in (("vectorial", vect), ("bm25", lex), ("memoria", mem)):
            for posicion, (doc, puntaje) in enumerate(lista, start=1):
                item = fusion.setdefault(doc.id, {"doc": doc, "rrf": 0.0})
                item["rrf"] += 1 / (RRF_K + posicion)
                item[f"score_{nombre}"] = puntaje

        mejores = sorted(fusion.values(), key=lambda x: x["rrf"], reverse=True)[:k]
        resultado = []
        for item in mejores:
            d = item.pop("doc")
            resultado.append(Document(page_content=d.page_content, metadata={**d.metadata, **item}, id=d.id))
        return resultado


def formatear_contexto(docs):
    """Arma el bloque de contexto numerado que va en el prompt."""
    bloques = []
    for i, d in enumerate(docs, start=1):
        m = d.metadata
        bloques.append(f"[{i}] Fuente: {m['archivo']} | {m['modelo']} | {m['seccion']}\n{d.page_content}")
    return "\n\n".join(bloques)

In [15]:
recuperador = RecuperadorHibrido(vectorstore)


def comparar(consulta, k=3):
    print("Consulta:", consulta)
    for modo in ("vectorial", "bm25", "hibrido"):
        print(f"\n  {modo}")
        for i, d in enumerate(recuperador.buscar(consulta, k=k, modo=modo), start=1):
            m = d.metadata
            print(f"   {i}. [{m['modelo']}] {m['seccion'][:50]:50s} "
                  f"vec={m.get('score_vectorial', '-')}  bm25={m.get('score_bm25', '-')}")
    print()


comparar("AL-205")
comparar("¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?")

Consulta: AL-205

  vectorial


   1. [TC-420] 5. Alarmas del control TVX-Control 8 > 5.1 Procedi vec=0.6863  bm25=-
   2. [TC-420] 5. Alarmas del control TVX-Control 8               vec=0.6436  bm25=-
   3. [TC-420] 5. Alarmas del control TVX-Control 8               vec=0.6175  bm25=-

  bm25
   1. [TC-420] 2. Seguridad                                       vec=-  bm25=7.9157
   2. [TC-420] 5. Alarmas del control TVX-Control 8               vec=-  bm25=7.7747
   3. [TC-420] 5. Alarmas del control TVX-Control 8 > 5.1 Procedi vec=-  bm25=7.5162

  hibrido


   1. [TC-420] 5. Alarmas del control TVX-Control 8 > 5.1 Procedi vec=0.6863  bm25=7.5162
   2. [TC-420] 5. Alarmas del control TVX-Control 8               vec=0.6436  bm25=7.7747
   3. [TC-420] 2. Seguridad                                       vec=0.6012  bm25=7.9157

Consulta: ¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?

  vectorial


   1. [CB-1200] 4. Mantenimiento > 4.1 Plan de mantenimiento       vec=0.7704  bm25=-
   2. [CB-1200] 5. Diagnóstico de fallas                           vec=0.6856  bm25=-
   3. [CB-1200] 5. Diagnóstico de fallas                           vec=0.6711  bm25=-

  bm25
   1. [CB-1200] 4. Mantenimiento > 4.4 Torques de ajuste de bulone vec=-  bm25=5.165
   2. [CB-1200] 1. Descripción general                             vec=-  bm25=4.8255
   3. [CB-1200] 4. Mantenimiento > 4.2 Tensado de la banda         vec=-  bm25=4.5514

  hibrido


   1. [CB-1200] 4. Mantenimiento > 4.4 Torques de ajuste de bulone vec=0.6689  bm25=5.165
   2. [CB-1200] 4. Mantenimiento > 4.1 Plan de mantenimiento       vec=0.7704  bm25=3.9124
   3. [CB-1200] 5. Diagnóstico de fallas                           vec=0.6856  bm25=3.5925



- **AL-205:** los tres métodos traen fragmentos del torno. El híbrido pone primero el procedimiento de la alarma (5.1), que está bien ubicado en las dos listas, por encima de la sección de seguridad que BM25 tenía primera.
- **"engrasan":** el manual dice "reengrasar", así que BM25 no trae el plan de mantenimiento entre los tres primeros: lo tapan otros fragmentos que repiten "rodamientos" y "cinta". La búsqueda vectorial lo pone primero, y en la híbrida queda segundo gracias a esa parte.

## 8. Toma de decisiones: en qué manual buscar

Antes de buscar, el sistema decide en qué manual hacerlo. Esto se resuelve sin el LLM, comparando la pregunta (en minúsculas, sin acentos y admitiendo plurales) con el código de modelo y los alias de cada manual. Es el método `detectar_manual` de la clase anterior.

| Situación | Decisión | Ejemplo |
|---|---|---|
| La pregunta nombra una máquina | Buscar solo en su manual | "la alarma A01 del compresor" → CT-75 |
| Pregunta de seguimiento sin máquina | Usar el manual de la pregunta anterior | "¿y cuántas veces puedo rearmarlo?" |
| No nombra ninguna, o nombra varias | Buscar en todos los manuales | "¿qué torque llevan los bulones M16?" |
| En la aplicación web el usuario elige un manual | Buscar solo en ese | filtro "Solo TC-420" |

Filtrar por manual evita mezclar datos de máquinas distintas.

In [16]:
for pregunta in ["¿Qué significa la alarma A01 del compresor?", "alarma del CT75", "¿Cuáles son los pasos del LOTO?",
                 "¿Qué torque llevan los bulones M16?", "¿Cómo bloqueo el torno?"]:
    print(f"{pregunta:45s} -> {recuperador.detectar_manual(pregunta) or 'todos los manuales'}")

¿Qué significa la alarma A01 del compresor?   -> CT-75
alarma del CT75                               -> CT-75
¿Cuáles son los pasos del LOTO?               -> PRO-SEG-001
¿Qué torque llevan los bulones M16?           -> todos los manuales
¿Cómo bloqueo el torno?                       -> todos los manuales


En "¿Cómo bloqueo el torno?" aparecen dos manuales (el del torno y el procedimiento de bloqueo), así que el sistema no filtra y busca en todos.

## 9. Cadena RAG

El prompt le pide al modelo que use solo la información del contexto, que cite cada dato con su número entre corchetes y que, si la respuesta no está, lo diga en lugar de inventar.

Para las preguntas de seguimiento ("¿y cuántas veces puedo rearmarlo?") no se hace una llamada extra al LLM: si la pregunta es corta o no nombra ninguna máquina, se le suma la pregunta anterior para buscar y se mantiene el manual de la conversación.

La cadena está armada con LCEL: plan (qué manual y qué buscar) → memoria de consultas validadas → recuperación → prompt → LLM. La consulta se convierte en embedding una sola vez y ese vector se usa para buscar en la memoria y en los manuales. La memoria se explica en la sección 11; hasta ahí la cadena se usa sin memoria, así que el bloque de consultas validadas del prompt queda en «(ninguna)».

In [17]:
MAX_HISTORIAL = 4  # mensajes anteriores que se le pasan al modelo
PALABRAS_SEGUIMIENTO = 5  # una pregunta así de corta se toma como continuación de la anterior
CITA = re.compile(r"\s*\[\d+\]")  # citas [n] de una respuesta
SYSTEM_PROMPT = """Sos un asistente para consultar manuales técnicos de máquinas \
industriales. Respondés en español, de forma clara y práctica, a operarios y técnicos.

Reglas:
1. Usá EXCLUSIVAMENTE la información del CONTEXTO (fragmentos de manuales). No uses \
conocimiento externo ni inventes valores.
2. Citá la fuente de cada dato con su número entre corchetes, por ejemplo [1] o [2][3].
3. Si el contexto no contiene la respuesta, decí: "No encontré esa información en los \
manuales cargados" y sugerí consultar al fabricante o al supervisor.
4. No mezcles datos de máquinas distintas.
5. Para procedimientos, usá pasos numerados. Mantené las unidades (bar, N·m, horas, °C).
6. Si la tarea implica intervenir la máquina y el contexto menciona medidas de seguridad \
(LOTO, despresurizar, EPP), recordalas al inicio.
7. Sé breve. No uses LaTeX ni fórmulas: escribí las cuentas en texto plano.
8. Las CONSULTAS VALIDADAS son preguntas parecidas que un técnico ya marcó como bien \
respondidas. Usalas como guía de qué información sirve y cómo responder, pero tomá los datos \
y las citas solo del CONTEXTO.

===== CONSULTAS VALIDADAS =====
{validadas}
===== CONTEXTO =====
{context}
===================="""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [18]:
def a_mensajes(historial):
    """Convierte [{'role': 'user'|'assistant', 'content': ...}] en mensajes de LangChain."""
    return [(HumanMessage if m["role"] == "user" else AIMessage)(content=m["content"])
            for m in historial[-MAX_HISTORIAL:]]


def cadena_respuesta(llm):
    return prompt | llm | StrOutputParser()


def preparar_consulta(pregunta, historial, recuperador, manual_elegido=None):
    """Decide en qué manual buscar y qué texto buscar, sin llamar al LLM.

    origen: "elegido" (lo fijó el usuario en la interfaz), "detectado" (la pregunta nombra
    la máquina), "contexto" (sale de la pregunta anterior) o None (se busca en todos).
    """
    anteriores = [m["content"] for m in historial if m["role"] == "user"]
    manual, origen = manual_elegido, "elegido" if manual_elegido else None
    detectado = recuperador.detectar_manual(pregunta)
    if not manual and detectado:
        manual, origen = detectado, "detectado"

    consulta = pregunta
    # Pregunta de seguimiento ("¿y cuántas veces puedo rearmarlo?"): se le suma la anterior
    # para que tenga palabras que buscar, y se mantiene el manual de la conversación.
    if anteriores and (not detectado or len(pregunta.split()) <= PALABRAS_SEGUIMIENTO):
        consulta = f"{anteriores[-1]} {pregunta}"
        if not manual:
            manual = recuperador.detectar_manual(anteriores[-1])
            origen = "contexto" if manual else None
    return {"consulta": consulta, "manual": manual, "origen": origen}


def recordar(plan, vector, memoria):
    """Busca consultas validadas parecidas. Si ninguna regla eligió el manual, usa el de la más parecida.

    Solo se usan recuerdos del mismo manual (o sin manual), para no mezclar máquinas.
    """
    recuerdos = memoria.buscar(vector) if memoria is not None else []
    if not plan["manual"] and recuerdos and recuerdos[0]["manual"]:
        plan = {**plan, "manual": recuerdos[0]["manual"], "origen": "aprendido"}
    if plan["manual"]:
        recuerdos = [r for r in recuerdos if r["manual"] in (plan["manual"], None)]
    return plan, recuerdos


def formatear_validadas(recuerdos):
    """Bloque de consultas validadas para el prompt. Se quitan sus citas [n], que eran de otro contexto."""
    if not recuerdos:
        return "(ninguna)"
    return "\n\n".join(f"Pregunta: {r['consulta']}\nRespuesta validada: {re.sub(CITA, '', r['respuesta'])}"
                       for r in recuerdos)


def recuperar(pregunta, historial, recuperador, memoria=None, k=TOP_K, manual_elegido=None):
    """Todo lo que pasa antes del LLM: plan, recuerdos de la memoria y fragmentos recuperados."""
    plan = preparar_consulta(pregunta, historial, recuperador, manual_elegido)
    vector = recuperador.embeber(plan["consulta"])  # un solo embedding para la memoria y los manuales
    plan, recuerdos = recordar(plan, vector, memoria)
    fuentes = recuperador.buscar(plan["consulta"], k=k, modelo=plan["manual"], vector=vector, recuerdos=recuerdos)
    return {"plan": plan, "recuerdos": recuerdos, "fuentes": fuentes}


def crear_cadena_rag(recuperador, llm, memoria=None, k=TOP_K):
    """Cadena completa. Entrada: {"input", "historial"}. Agrega "plan", "recuerdos", "fuentes" y "respuesta"."""
    return (
        RunnableLambda(lambda x: {**x, **recuperar(x["input"], x["historial"], recuperador, memoria, k)})
        | RunnablePassthrough.assign(
            context=lambda x: formatear_contexto(x["fuentes"]),
            validadas=lambda x: formatear_validadas(x["recuerdos"]),
            chat_history=lambda x: a_mensajes(x["historial"]))
        | RunnablePassthrough.assign(respuesta=cadena_respuesta(llm))
    )

In [19]:
llm = get_llm()
cadena_rag = crear_cadena_rag(recuperador, llm)
historial = []


def preguntar(pregunta, usar_historial=True):
    salida = cadena_rag.invoke({"input": pregunta, "historial": historial if usar_historial else []})
    plan = salida["plan"]
    print(f"Manual: {plan['manual'] or 'todos'} ({plan['origen'] or 'sin filtro'})")
    print(f"Texto buscado: {plan['consulta']}\n")
    print(salida["respuesta"])
    print("\nFuentes:")
    for i, d in enumerate(salida["fuentes"], start=1):
        print(f"  [{i}] {d.metadata['modelo']} - {d.metadata['seccion']}")
    if usar_historial:
        historial.extend([{"role": "user", "content": pregunta},
                          {"role": "assistant", "content": salida["respuesta"]}])


preguntar("¿Qué significa la alarma A01 del compresor y cómo la soluciono?")

Manual: CT-75 (detectado)
Texto buscado: ¿Qué significa la alarma A01 del compresor y cómo la soluciono?

La alarma **A01** en el compresor AERON CT-75 indica una **temperatura de descarga alta** [2]. La parada del equipo por esta alarma ocurre al alcanzar los 110 °C [1].

**Medidas de seguridad:**
Antes de intervenir, asegúrese de que el equipo esté detenido y siga los procedimientos de seguridad de su planta (LOTO).

**Acciones correctivas:**
Para solucionar esta alarma, verifique los siguientes puntos según el manual [2]:

1. **Nivel de aceite:** Verifique que el nivel en el visor del tanque separador esté entre las marcas MIN y MAX [4]. Si es bajo, rellene.
2. **Radiador:** Inspeccione si el radiador está obstruido y límpielo si es necesario.
3. **Ventilación:** Asegúrese de que la temperatura ambiente de la sala no supere los 40 °C. Si es superior, instale ductos de extracción de aire caliente [1].
4. **Válvula termostática:** Revise si la válvula termostática está trabada.

**Not

Pregunta de seguimiento: no nombra la máquina ni la alarma.

In [20]:
preguntar("¿Y cuántas veces seguidas puedo rearmarlo?")

Manual: CT-75 (contexto)
Texto buscado: ¿Qué significa la alarma A01 del compresor y cómo la soluciono? ¿Y cuántas veces seguidas puedo rearmarlo?

No debe rearmar el equipo más de **dos veces consecutivas** sin haber identificado la causa del problema [1]. 

El funcionamiento repetido a alta temperatura degrada el aceite y puede causar daños en los rodamientos del elemento [1].

Fuentes:
  [1] CT-75 - 5. Alarmas y diagnóstico > 5.1 Detalle de la alarma A01
  [2] CT-75 - 5. Alarmas y diagnóstico
  [3] CT-75 - 5. Alarmas y diagnóstico > 5.2 Problemas frecuentes sin código de alarma
  [4] CT-75 - 3. Operación > 3.1 Arranque


Pregunta sobre algo que no está en los manuales: el sistema tiene que decir que no encontró la información.

In [21]:
preguntar("¿Cuál es el torque de la tapa de cilindros del motor diésel XR-9?", usar_historial=False)

Manual: todos (sin filtro)
Texto buscado: ¿Cuál es el torque de la tapa de cilindros del motor diésel XR-9?

No encontré esa información en los manuales cargados. Te sugiero consultar al fabricante o a tu supervisor para obtener el manual técnico específico del motor XR-9.

Fuentes:
  [1] TC-420 - 4. Mantenimiento preventivo > 4.4 Torques de ajuste
  [2] CB-1200 - 4. Mantenimiento > 4.4 Torques de ajuste de bulonería (grado 8.8)
  [3] CT-75 - 4. Mantenimiento preventivo > 4.3 Procedimiento de cambio de aceite
  [4] IP-250 - 3. Operación > 3.3 Cambio de molde


## 10. Evaluación con casos de prueba

Armamos 14 preguntas con respuesta conocida: 6 que nombran códigos o medidas (`A01`, `E-17`, `M16`), 7 semánticas (el problema descripto con otras palabras) y 1 sobre algo que no está en ningún manual. Evaluamos:

- **Recuperación:** Hit@k (proporción de preguntas en las que algún chunk del top-k tiene la respuesta) y MRR (promedio de 1/posición del primer chunk correcto). Comparamos búsqueda vectorial, BM25 e híbrida.
- **Detección del manual:** si el sistema eligió bien el manual.
- **Respuesta final:** si la respuesta contiene el dato esperado o, en la pregunta fuera de los manuales, si dice que no lo encontró.

In [22]:
@dataclass
class Caso:
    pregunta: str
    modelo: str | None  # manual donde está la respuesta (None = fuera de dominio)
    evidencia: str      # texto que tiene que estar en el chunk correcto
    respuesta: str      # dato que tiene que aparecer en la respuesta
    tipo: str           # "código", "semántica" o "fuera de dominio"


CASOS = [
    Caso("¿Qué significa la alarma A01 del compresor?", "CT-75", "A01", "temperatura", "código"),
    Caso("¿Cada cuántas horas hay que cambiar el aceite del compresor?", "CT-75", "4.000 horas", "4.000", "semántica"),
    Caso("¿Qué aceite usa el CT-75 y cuántos litros lleva?", "CT-75", "LubriScrew", "38", "código"),
    Caso("Tengo la alarma AL-205 en el torno", "TC-420", "AL-205", "40 bar", "código"),
    Caso("El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sistema",
         "TC-420", "presión hidráulica baja", "hidráulic", "semántica"),
    Caso("¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?", "TC-420", "1.000 rpm", "1.000", "semántica"),
    Caso("¿Qué concentración tiene que tener el refrigerante del torno?", "TC-420", "5 % a 8 %", "8", "semántica"),
    Caso("La inyectora marca E-17, ¿qué hago?", "IP-250", "termocupla", "termocupla", "código"),
    Caso("¿A qué temperatura se inyecta el ABS?", "IP-250", "ABS", "210", "código"),
    Caso("¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?", "CB-1200", "NLGI 2", "500", "semántica"),
    Caso("¿Qué torque llevan los bulones M16 de la cinta?", "CB-1200", "M16", "210", "código"),
    Caso("La banda resbala sobre el tambor que la mueve, ¿qué puede ser?", "CB-1200", "patina", "tensi", "semántica"),
    Caso("Un compañero se fue a su casa y dejó su candado puesto en la máquina, ¿quién lo puede sacar?",
         "PRO-SEG-001", "Candado olvidado", "jefe de mantenimiento", "semántica"),
    Caso("¿Cuál es el torque de la tapa de cilindros del motor diésel XR-9?", None, "", "no encontr", "fuera de dominio"),
]

In [23]:
pd.DataFrame([vars(c) for c in CASOS])[["pregunta", "modelo", "tipo", "evidencia"]]

,pregunta,modelo,tipo,evidencia
0,¿Qué significa la alarma A01 del compresor?,CT-75,código,A01
1,¿Cada cuántas horas hay que cambiar el aceite del compresor?,CT-75,semántica,4.000 horas
2,¿Qué aceite usa el CT-75 y cuántos litros lleva?,CT-75,código,LubriScrew
3,Tengo la alarma AL-205 en el torno,TC-420,código,AL-205
4,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis...",TC-420,semántica,presión hidráulica baja
5,¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?,TC-420,semántica,1.000 rpm
6,¿Qué concentración tiene que tener el refrigerante del torno?,TC-420,semántica,5 % a 8 %
7,"La inyectora marca E-17, ¿qué hago?",IP-250,código,termocupla
8,¿A qué temperatura se inyecta el ABS?,IP-250,código,ABS
9,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?,CB-1200,semántica,NLGI 2


In [24]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def es_relevante(doc, caso):
    return doc.metadata["modelo"] == caso.modelo and normalizar(caso.evidencia) in normalizar(doc.page_content)


def evaluar_recuperacion(recuperador, k, modos=("vectorial", "bm25", "hibrido")):
    """Una fila por (pregunta, modo) con la posición del primer chunk relevante."""
    filas = []
    for caso in CASOS:
        if caso.modelo is None:
            continue
        for modo in modos:
            docs = recuperador.buscar(caso.pregunta, k=k, modo=modo)
            posicion = next((i for i, d in enumerate(docs, start=1) if es_relevante(d, caso)), None)
            filas.append({"pregunta": caso.pregunta, "tipo": caso.tipo, "modo": modo, "posicion": posicion,
                          "hit": posicion is not None, "rr": 1 / posicion if posicion else 0.0})
    return pd.DataFrame(filas)


def resumen_recuperacion(df):
    """Hit@k y MRR por modo, en total y por tipo de pregunta."""
    total = df.groupby("modo").agg(hit_at_k=("hit", "mean"), mrr=("rr", "mean"))
    por_tipo = df.pivot_table(index="modo", columns="tipo", values="hit", aggfunc="mean")
    por_tipo.columns = [f"hit@k ({c})" for c in por_tipo.columns]
    return total.join(por_tipo).round(3).loc[["vectorial", "bm25", "hibrido"]]


def evaluar_deteccion(recuperador):
    """correcto = eligió el manual de la respuesta; todos = no filtró; incorrecto = eligió otro."""
    filas = []
    for caso in CASOS:
        detectado = recuperador.detectar_manual(caso.pregunta)
        if detectado is None:
            resultado = "todos"
        else:
            resultado = "correcto" if detectado == caso.modelo else "incorrecto"
        filas.append({"pregunta": caso.pregunta, "esperado": caso.modelo or "(ninguno)",
                      "detectado": detectado or "(todos)", "resultado": resultado})
    return pd.DataFrame(filas)


def evaluar_respuestas(cadena_rag, pausa=0, casos=CASOS):
    """Corre la cadena RAG con cada caso y verifica la respuesta.

    pausa: segundos entre preguntas, por el límite de pedidos por minuto del plan gratuito.
    """
    filas = []
    for caso in casos:
        inicio = time.perf_counter()
        salida = cadena_rag.invoke({"input": caso.pregunta, "historial": []})
        segundos = time.perf_counter() - inicio
        correcta = normalizar(caso.respuesta) in normalizar(salida["respuesta"])
        filas.append({"pregunta": caso.pregunta, "tipo": caso.tipo, "correcta": correcta,
                      "segundos": round(segundos, 1), "recuerdos": len(salida["recuerdos"]),
                      "respuesta": salida["respuesta"]})
        time.sleep(pausa)
    return pd.DataFrame(filas)

### 10.1 Recuperación

Hipótesis: BM25 funciona mejor en las preguntas con códigos, la búsqueda vectorial en las semánticas, y la híbrida es la más pareja.

In [25]:
df_rec = evaluar_recuperacion(recuperador, k=TOP_K)
print(f"k = {TOP_K}")
resumen_recuperacion(df_rec)

k = 4


,hit_at_k,mrr,hit@k (código),hit@k (semántica)
modo,,,,
vectorial,1.000,0.923,1.0,1.000
bm25,0.846,0.808,1.0,0.714
hibrido,0.923,0.885,1.0,0.857


In [26]:
# Posición del chunk correcto en cada pregunta (vacío = no apareció en el top-k)
df_rec.pivot_table(index=["tipo", "pregunta"], columns="modo", values="posicion", aggfunc="first")[
    ["vectorial", "bm25", "hibrido"]]

modo                                                                                                            vectorial  \
tipo      pregunta                                                                                                          
código    La inyectora marca E-17, ¿qué hago?                                                                         1.0   
          Tengo la alarma AL-205 en el torno                                                                          1.0   
          ¿A qué temperatura se inyecta el ABS?                                                                       1.0   
          ¿Qué aceite usa el CT-75 y cuántos litros lleva?                                                            2.0   
          ¿Qué significa la alarma A01 del compresor?                                                                 1.0   
          ¿Qué torque llevan los bulones M16 de la cinta?                                                             1.0   
semántica El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sistema        2.0   
          La banda resbala sobre el tambor que la mueve, ¿qué puede ser?                                              1.0   
          Un compañero se fue a su casa y dejó su candado puesto en la máquina, ¿quién lo puede sacar?                1.0   
          ¿Cada cuántas horas hay que cambiar el aceite del compresor?                                                1.0   
          ¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?                                        1.0   
          ¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?                                1.0   
          ¿Qué concentración tiene que tener el refrigerante del torno?                                               1.0   

modo                                                                                                            bm25  \
tipo      pregunta                                                                                                     
código    La inyectora marca E-17, ¿qué hago?                                                                    1.0   
          Tengo la alarma AL-205 en el torno                                                                     1.0   
          ¿A qué temperatura se inyecta el ABS?                                                                  1.0   
          ¿Qué aceite usa el CT-75 y cuántos litros lleva?                                                       1.0   
          ¿Qué significa la alarma A01 del compresor?                                                            1.0   
          ¿Qué torque llevan los bulones M16 de la cinta?                                                        1.0   
semántica El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sistema   NaN   
          La banda resbala sobre el tambor que la mueve, ¿qué puede ser?                                         1.0   
          Un compañero se fue a su casa y dejó su candado puesto en la máquina, ¿quién lo puede sacar?           2.0   
          ¿Cada cuántas horas hay que cambiar el aceite del compresor?                                           1.0   
          ¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?                                   NaN   
          ¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?                           1.0   
          ¿Qué concentración tiene que tener el refrigerante del torno?                                          1.0   

modo                                                                                                            hibrido  
tipo      pregunta                                                                                                       
código    La inyectora marca E-17, ¿qué hago?                                  

**Análisis.** La hipótesis se cumplió solo en parte:

- **BM25** encontró todas las preguntas de código, pero falló en 2 de las 7 semánticas: "¿cada cuánto se engrasan…?" (el manual dice "reengrasar") y "el torno se queda sin fuerza… parece falta de aceite" (el manual habla de "presión hidráulica baja").
- **La búsqueda vectorial** encontró las 13, incluidas las de código. En este set cada código aparece en fragmentos que además hablan del tema, así que los embeddings no se confundieron. El problema de la sección 5 (AL-205 contra AL-250) aparecería con códigos parecidos entre sí, y acá no hay.
- **La híbrida** mejoró a BM25 (Hit@4 de 0,846 a 0,923), pero quedó por debajo de la vectorial. Falló en "el torno se queda sin fuerza…": la vectorial tenía el fragmento correcto en la posición 2, pero BM25 no lo encontró y subió otros fragmentos del torno que, al fusionar, quedaron adelante.

Con 13 preguntas no alcanza para decir que un método es mejor en general. Mantuvimos la búsqueda híbrida porque protege las consultas por código exacto, que en una planta real, con cientos de códigos parecidos, son las más riesgosas para los embeddings. Además, en esa pregunta la respuesta final fue correcta igual (sección 10.3), porque otros fragmentos del torno recuperados también hablan de la presión hidráulica.

Esta evaluación busca en todos los manuales. En el sistema completo se agrega el filtro por manual de la sección 8.

### 10.2 Detección del manual

"todos" significa que el sistema no filtró y buscó en todos los manuales: no es un error, porque la búsqueda igual puede encontrar el fragmento. Sería un error elegir un manual equivocado, porque la respuesta quedaría afuera de la búsqueda.

In [27]:
df_det = evaluar_deteccion(recuperador)
print(df_det["resultado"].value_counts().to_string())
df_det

resultado
correcto    10
todos        4


,pregunta,esperado,detectado,resultado
0,¿Qué significa la alarma A01 del compresor?,CT-75,CT-75,correcto
1,¿Cada cuántas horas hay que cambiar el aceite del compresor?,CT-75,CT-75,correcto
2,¿Qué aceite usa el CT-75 y cuántos litros lleva?,CT-75,CT-75,correcto
3,Tengo la alarma AL-205 en el torno,TC-420,TC-420,correcto
4,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis...",TC-420,TC-420,correcto
5,¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?,TC-420,(todos),todos
6,¿Qué concentración tiene que tener el refrigerante del torno?,TC-420,TC-420,correcto
7,"La inyectora marca E-17, ¿qué hago?",IP-250,IP-250,correcto
8,¿A qué temperatura se inyecta el ABS?,IP-250,(todos),todos
9,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?,CB-1200,CB-1200,correcto


### 10.3 Respuestas del sistema

Se corre la cadena completa con las 14 preguntas (14 llamadas al LLM, con una pausa de 4 segundos entre cada una por el límite de pedidos por minuto del plan gratuito).

In [28]:
df_resp = evaluar_respuestas(cadena_rag, pausa=4)
print(f"Respuestas correctas: {df_resp['correcta'].sum()} de {len(df_resp)} ({df_resp['correcta'].mean():.0%})")
print(f"Tiempo promedio por respuesta: {df_resp['segundos'].mean():.1f} s")
display(df_resp.groupby("tipo")["correcta"].mean().rename("exactitud").to_frame())
df_resp[["tipo", "pregunta", "correcta", "respuesta"]]

Respuestas correctas: 14 de 14 (100%)
Tiempo promedio por respuesta: 9.4 s


,exactitud
tipo,
código,1.0
fuera de dominio,1.0
semántica,1.0


,tipo,pregunta,correcta,respuesta
0,código,¿Qué significa la alarma A01 del compresor?,True,La alarma **A01** en el compresor AERON CT-75 indica una **temperatura de descarga alta** [2].\n...
1,semántica,¿Cada cuántas horas hay que cambiar el aceite del compresor?,True,El cambio de aceite completo del compresor AERON CT-75 debe realizarse cada 4.000 horas de funci...
2,código,¿Qué aceite usa el CT-75 y cuántos litros lleva?,True,"Para el compresor AERON CT-75, se debe utilizar exclusivamente aceite sintético **AERON LubriScr..."
3,código,Tengo la alarma AL-205 en el torno,True,"Para intervenir la máquina ante la alarma **AL-205 (Presión hidráulica baja)**, tené en cuenta l..."
4,semántica,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis...",True,"Para intervenir el torno TC-420 ante una posible falla de presión hidráulica, es obligatorio seg..."
5,semántica,¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?,True,Para realizar la rutina de calentamiento del husillo en el torno CNC TORVEX TC-420 tras una para...
6,semántica,¿Qué concentración tiene que tener el refrigerante del torno?,True,La concentración correcta del refrigerante para el torno TC-420 debe ser de entre el 5 % y el 8 ...
7,código,"La inyectora marca E-17, ¿qué hago?",True,El error **E-17** en la inyectora PLASTIMAQ IP-250 indica una falla de termocupla (lectura abier...
8,código,¿A qué temperatura se inyecta el ABS?,True,"Para procesar ABS en la inyectora PLASTIMAQ IP-250, se deben considerar los siguientes parámetro..."
9,semántica,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?,True,Los rodamientos de los tambores de la cinta transportadora CB-1200 deben reengrasarse cada **500...


**Análisis.** Las 14 respuestas contienen el dato esperado, incluida la pregunta fuera de los manuales, donde el sistema dijo que no encontró la información en lugar de inventar un torque. Cada respuesta tardó en promedio unos 9 segundos, casi todo en la llamada al modelo.

Hay que tener en cuenta que el criterio es simple: se verifica que un dato clave aparezca en el texto (por ejemplo "210" o "termocupla"), no que toda la respuesta sea correcta, así que conviene leer las respuestas de la tabla y no quedarse solo con el porcentaje. Corrimos la evaluación dos veces y las dos dio 14 de 14; con temperatura 0.1 cambia algo la redacción, pero no los datos.

## 11. Aprendizaje a partir de las conversaciones

El sistema aprende de lo que conversa, pero **no guarda las conversaciones**. Cuando el técnico marca una respuesta como correcta (👍), se guarda solo esa consulta: el texto que se buscó, la respuesta y los fragmentos del manual que la respuesta citó. Todo eso queda en una colección aparte de Chroma (`memoria`), que persiste en disco. Con 👎 no se aprende nada, y si la consulta ya estaba en la memoria se quita.

Ante una pregunta nueva, se buscan en la memoria las consultas validadas parecidas (similitud coseno ≥ `UMBRAL_MEMORIA`) y se usan de tres formas:

1. **Elegir el manual:** si ninguna regla lo decidió (la pregunta no nombra la máquina ni hay contexto), se usa el manual de la consulta validada más parecida (origen «aprendido»). Ese manual es el de los fragmentos que citó la respuesta validada.
2. **Recuperación:** los fragmentos que citó la respuesta validada entran en RRF como una **tercera lista**, junto a la vectorial y BM25.
3. **Prompt:** la pregunta y la respuesta validadas se le pasan al LLM como ejemplo (regla 8). Las citas [n] se quitan, porque eran de otro contexto, y los datos se siguen tomando solo de los fragmentos recuperados.

Solo se aprende lo validado para que el sistema no repita una respuesta equivocada. Tampoco hace falta reentrenar el modelo: lo aprendido se aplica desde la consulta siguiente y se puede deshacer.

Las preguntas se guardan con el embedding de tipo consulta (`RETRIEVAL_QUERY`), el mismo que se calcula para cada pregunta nueva, porque acá se compara una pregunta con otra y no una pregunta con un fragmento de manual.

In [29]:
COLECCION_MEMORIA = "memoria"


def fragmentos_citados(respuesta, fuentes):
    """Fuentes que la respuesta citó con [n], en orden de aparición y sin repetir (None = fuente que ya no existe)."""
    citadas = []
    for n in re.findall(r"\[(\d+)\]", respuesta):
        i = int(n) - 1
        if 0 <= i < len(fuentes) and fuentes[i] and fuentes[i] not in citadas:
            citadas.append(fuentes[i])
    return citadas


class Memoria:
    def __init__(self, vectorstore, umbral=UMBRAL_MEMORIA, maximo=MAX_RECUERDOS):
        self.vs = vectorstore
        self.umbral = umbral
        self.maximo = maximo

    def __len__(self):
        return len(self.vs.get(include=[])["ids"])

    def aprender(self, id, consulta, respuesta, fuentes):
        """Guarda una consulta validada. fuentes: [{"id", "modelo"}] en el orden en que se enviaron al LLM.

        El manual aprendido es el de los fragmentos citados, si son todos del mismo.
        """
        citadas = fragmentos_citados(respuesta, fuentes)
        modelos = {f["modelo"] for f in citadas}
        self.vs.add_documents([Document(page_content=consulta, metadata={
            "respuesta": respuesta,
            "manual": modelos.pop() if len(modelos) == 1 else "",
            "fragmentos": ",".join(f["id"] for f in citadas),
            "fecha": datetime.now().isoformat(timespec="seconds"),
        })], ids=[id])
        return self.obtener(id)

    def olvidar(self, ids):
        if ids:
            self.vs.delete(ids=list(ids))

    @staticmethod
    def _recuerdo(id, texto, m, similitud=None):
        return {"id": id, "consulta": texto, "respuesta": m["respuesta"], "manual": m["manual"] or None,
                "fragmentos": [f for f in m["fragmentos"].split(",") if f], "fecha": m["fecha"],
                "similitud": similitud}

    def obtener(self, id):
        datos = self.vs.get(ids=[id], include=["documents", "metadatas"])
        if not datos["ids"]:
            return None
        return self._recuerdo(id, datos["documents"][0], datos["metadatas"][0])

    def listar(self):
        datos = self.vs.get(include=["documents", "metadatas"])
        recuerdos = [self._recuerdo(i, t, m) for i, t, m in zip(datos["ids"], datos["documents"], datos["metadatas"])]
        return sorted(recuerdos, key=lambda r: r["fecha"], reverse=True)

    def buscar(self, vector):
        """Consultas validadas parecidas (similitud coseno >= umbral), de la más a la menos parecida."""
        cantidad = len(self)
        if not cantidad:
            return []
        resultados = self.vs.similarity_search_by_vector_with_relevance_scores(vector, k=min(self.maximo, cantidad))
        recuerdos = [self._recuerdo(d.id, d.page_content, d.metadata, round(1 - dist, 4)) for d, dist in resultados]
        return [r for r in recuerdos if r["similitud"] >= self.umbral]


def obtener_memoria(nombre=COLECCION_MEMORIA, persistir=True):
    # Las preguntas se guardan con el mismo tipo de embedding que las consultas (RETRIEVAL_QUERY),
    # porque acá se compara una pregunta con otra pregunta, no con un fragmento de manual
    vs = Chroma(collection_name=nombre, embedding_function=get_embeddings("RETRIEVAL_QUERY"),
                persist_directory=str(CHROMA_DIR) if persistir else None,
                collection_metadata={"hnsw:space": "cosine"})
    return Memoria(vs)

### 11.1 Umbral de similitud

El umbral decide cuándo dos preguntas son «la misma consulta». Si es muy bajo, se reutilizan consultas que solo hablan de un tema parecido; si es muy alto, no se reconoce la misma pregunta dicha con otras palabras.

Para elegirlo armamos 7 consultas «ya validadas», que repiten con otras palabras 7 de los casos de prueba, y medimos la similitud de cada una con los 14 casos.

In [30]:
# Consultas que un técnico ya hizo antes con otras palabras y validó con 👍 (se usan para entrenar la memoria)
VALIDADAS = [
    Caso("¿Qué temperatura de inyección lleva el ABS?", "IP-250", "ABS", "210", "validada"),
    Caso("La banda patina en el tambor motriz, ¿qué reviso?", "CB-1200", "patina", "tensi", "validada"),
    Caso("La máquina estuvo parada tres días, ¿cómo hago el calentamiento del husillo?", "TC-420", "1.000 rpm",
         "1.000", "validada"),
    Caso("Al torno le falta fuerza en el plato para agarrar la pieza, ¿qué puede ser?", "TC-420",
         "presión hidráulica baja", "hidráulic", "validada"),
    Caso("¿Cada cuánto hay que engrasar los rodamientos de la cinta?", "CB-1200", "NLGI 2", "500", "validada"),
    Caso("Un operario se fue y dejó su candado colocado, ¿quién puede retirarlo?", "PRO-SEG-001", "Candado olvidado",
         "jefe de mantenimiento", "validada"),
    Caso("¿Cuántos litros de aceite lleva el compresor y de qué tipo?", "CT-75", "LubriScrew", "38", "validada"),
]


emb_consultas = get_embeddings("RETRIEVAL_QUERY")
textos = [c.pregunta for c in VALIDADAS] + [c.pregunta for c in CASOS]
v = np.array(emb_consultas.embed_documents(textos))
v = v / np.linalg.norm(v, axis=1, keepdims=True)
sim = v[:len(VALIDADAS)] @ v[len(VALIDADAS):].T

df_sim = pd.DataFrame([{"similitud": round(float(sim[i, j]), 3), "validada": val.pregunta, "caso": caso.pregunta,
                        "misma consulta": caso.modelo == val.modelo and caso.evidencia == val.evidencia}
                       for i, val in enumerate(VALIDADAS) for j, caso in enumerate(CASOS)])
print(f"Umbral: {UMBRAL_MEMORIA}\n\nLa misma consulta con otras palabras:")
display(df_sim[df_sim["misma consulta"]].sort_values("similitud")[["similitud", "validada", "caso"]])
print("Consultas distintas, las 3 más parecidas:")
df_sim[~df_sim["misma consulta"]].nlargest(3, "similitud")[["similitud", "validada", "caso"]]

Umbral: 0.88

La misma consulta con otras palabras:


,similitud,validada,caso
86,0.741,¿Cuántos litros de aceite lleva el compresor y de qué tipo?,¿Qué aceite usa el CT-75 y cuántos litros lleva?
46,0.899,"Al torno le falta fuerza en el plato para agarrar la pieza, ¿qué puede ser?","El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis..."
25,0.917,"La banda patina en el tambor motriz, ¿qué reviso?","La banda resbala sobre el tambor que la mueve, ¿qué puede ser?"
82,0.920,"Un operario se fue y dejó su candado colocado, ¿quién puede retirarlo?","Un compañero se fue a su casa y dejó su candado puesto en la máquina, ¿quién lo puede sacar?"
33,0.950,"La máquina estuvo parada tres días, ¿cómo hago el calentamiento del husillo?",¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?
65,0.951,¿Cada cuánto hay que engrasar los rodamientos de la cinta?,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?
8,0.968,¿Qué temperatura de inyección lleva el ABS?,¿A qué temperatura se inyecta el ABS?


Consultas distintas, las 3 más parecidas:


,similitud,validada,caso
85,0.867,¿Cuántos litros de aceite lleva el compresor y de qué tipo?,¿Cada cuántas horas hay que cambiar el aceite del compresor?
48,0.693,"Al torno le falta fuerza en el plato para agarrar la pieza, ¿qué puede ser?",¿Qué concentración tiene que tener el refrigerante del torno?
88,0.693,¿Cuántos litros de aceite lleva el compresor y de qué tipo?,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis..."


**Análisis.** Cuando es la misma consulta con otras palabras, la similitud queda entre 0,899 y 0,968, salvo un caso. Entre consultas distintas, la más alta es 0,867: «¿cuántos litros de aceite lleva el compresor?» contra «¿cada cuántas horas se cambia el aceite del compresor?», que son la misma máquina y el mismo tema, pero piden datos diferentes. Por eso el umbral quedó en 0,88, entre esos dos valores. Con 0,85 se habría reutilizado la respuesta sobre los litros para una pregunta sobre el intervalo.

La excepción es la pregunta del aceite del CT-75 (0,741): la validada nombra «el compresor» y el caso nombra el código «CT-75», y los embeddings no los consideran tan parecidos. Con este umbral esa consulta no se reconoce, así que el umbral prioriza no equivocarse antes que reconocer todas las repeticiones.

### 11.2 El técnico valida respuestas

Simulamos al técnico: se hace cada una de las 7 consultas y, si la respuesta tiene el dato correcto, se marca con 👍 y se aprende. La memoria de esta prueba es temporal (en RAM), así que no modifica la del server.

In [31]:
# --- Aprendizaje ---
def entrenar_memoria(cadena_rag, memoria, pausa=0, casos=VALIDADAS):
    """Simula al técnico: corre cada consulta y la valida (👍) solo si la respuesta tiene el dato correcto."""
    filas = []
    for i, caso in enumerate(casos, start=1):
        salida = cadena_rag.invoke({"input": caso.pregunta, "historial": []})
        validada = normalizar(caso.respuesta) in normalizar(salida["respuesta"])
        recuerdo = None
        if validada:
            fuentes = [{"id": d.id, "modelo": d.metadata["modelo"]} for d in salida["fuentes"]]
            recuerdo = memoria.aprender(f"validada-{i:02d}", salida["plan"]["consulta"], salida["respuesta"], fuentes)
        filas.append({"pregunta": caso.pregunta, "validada": validada,
                      "manual aprendido": recuerdo["manual"] if recuerdo else None,
                      "fragmentos citados": ", ".join(recuerdo["fragmentos"]) if recuerdo else ""})
        time.sleep(pausa)
    return pd.DataFrame(filas)


def _resultado_manual(manual, esperado):
    if manual is None:
        return "todos"
    return "correcto" if manual == esperado else "incorrecto"


def evaluar_memoria(recuperador, memoria, k):
    """Para cada caso: manual elegido y posición del chunk correcto, sin y con la memoria.

    La recuperación se compara sin filtro por manual, igual que en evaluar_recuperacion.
    """
    filas = []
    for caso in CASOS:
        plan = preparar_consulta(caso.pregunta, [], recuperador)
        vector = recuperador.embeber(plan["consulta"])
        plan_con, recuerdos = recordar(plan, vector, memoria)
        posiciones = []
        for rec in ([], recuerdos):
            docs = recuperador.buscar(caso.pregunta, k=k, vector=vector, recuerdos=rec)
            posiciones.append(next((i for i, d in enumerate(docs, start=1) if es_relevante(d, caso)), None))
        filas.append({
            "pregunta": caso.pregunta, "tipo": caso.tipo, "esperado": caso.modelo or "(ninguno)",
            "recuerdos": len(recuerdos), "similitud": max((r["similitud"] for r in recuerdos), default=None),
            "manual sin memoria": _resultado_manual(plan["manual"], caso.modelo),
            "manual con memoria": _resultado_manual(plan_con["manual"], caso.modelo),
            "posición sin memoria": posiciones[0], "posición con memoria": posiciones[1],
        })
    return pd.DataFrame(filas)


def resumen_memoria(df):
    """Detección del manual y Hit@k / MRR (sobre los casos con respuesta en los manuales), sin y con memoria."""
    con_respuesta = df[df["esperado"] != "(ninguno)"]
    filas = {}
    for cuando in ("sin memoria", "con memoria"):
        conteo = df[f"manual {cuando}"].value_counts()
        pos = con_respuesta[f"posición {cuando}"]
        filas[cuando] = {"manual correcto": conteo.get("correcto", 0), "todos": conteo.get("todos", 0),
                         "manual incorrecto": conteo.get("incorrecto", 0), "hit@k": round(pos.notna().mean(), 3),
                         "mrr": round((1 / pos).fillna(0).mean(), 3)}
    return pd.DataFrame(filas).T

In [32]:
memoria_eval = obtener_memoria("memoria_evaluacion", persistir=False)
cadena_rag_mem = crear_cadena_rag(recuperador, llm, memoria_eval)

df_ent = entrenar_memoria(cadena_rag_mem, memoria_eval, pausa=4)
print(f"Consultas aprendidas: {len(memoria_eval)} de {len(VALIDADAS)}")
df_ent

Consultas aprendidas: 7 de 7


,pregunta,validada,manual aprendido,fragmentos citados
0,¿Qué temperatura de inyección lleva el ABS?,True,IP-250,IP-250_inyectora_plastimaq.md::003
1,"La banda patina en el tambor motriz, ¿qué reviso?",True,CB-1200,"CB-1200_cinta_transportadora_transbelt.md::008, CB-1200_cinta_transportadora_transbelt.md::005"
2,"La máquina estuvo parada tres días, ¿cómo hago el calentamiento del husillo?",True,TC-420,TC-420_torno_cnc_torvex.md::003
3,"Al torno le falta fuerza en el plato para agarrar la pieza, ¿qué puede ser?",True,TC-420,"TC-420_torno_cnc_torvex.md::002, TC-420_torno_cnc_torvex.md::011"
4,¿Cada cuánto hay que engrasar los rodamientos de la cinta?,True,CB-1200,CB-1200_cinta_transportadora_transbelt.md::004
5,"Un operario se fue y dejó su candado colocado, ¿quién puede retirarlo?",True,PRO-SEG-001,PRO-SEG-001_bloqueo_etiquetado_LOTO.md::006
6,¿Cuántos litros de aceite lleva el compresor y de qué tipo?,True,CT-75,"CT-75_compresor_tornillo_aeron.md::007, CT-75_compresor_tornillo_aeron.md::001"


**Análisis.** Las 7 respuestas tenían el dato correcto, así que las 7 se aprendieron. En todas, el manual aprendido es el correcto: sale de los fragmentos que citó cada respuesta, aunque la pregunta no nombre la máquina (por ejemplo, la del ABS citó solo el fragmento de temperaturas de la IP-250).

### 11.3 ¿Qué cambió con lo aprendido?

Se vuelven a evaluar los 14 casos de prueba, sin y con la memoria: qué manual se elige y en qué posición queda el fragmento correcto (búsqueda híbrida sin filtro por manual, como en la sección 10.1).

In [33]:
df_mem = evaluar_memoria(recuperador, memoria_eval, k=TOP_K)
display(resumen_memoria(df_mem))
df_mem

,manual correcto,todos,manual incorrecto,hit@k,mrr
sin memoria,10.0,4.0,0.0,0.923,0.885
con memoria,13.0,1.0,0.0,0.923,0.923


,pregunta,tipo,esperado,recuerdos,similitud,manual sin memoria,manual con memoria,posición sin memoria,posición con memoria
0,¿Qué significa la alarma A01 del compresor?,código,CT-75,0,NaN,correcto,correcto,1.0,1.0
1,¿Cada cuántas horas hay que cambiar el aceite del compresor?,semántica,CT-75,0,NaN,correcto,correcto,1.0,1.0
2,¿Qué aceite usa el CT-75 y cuántos litros lleva?,código,CT-75,0,NaN,correcto,correcto,1.0,1.0
3,Tengo la alarma AL-205 en el torno,código,TC-420,0,NaN,correcto,correcto,1.0,1.0
4,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis...",semántica,TC-420,1,0.8994,correcto,correcto,NaN,NaN
5,¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?,semántica,TC-420,1,0.9495,todos,correcto,1.0,1.0
6,¿Qué concentración tiene que tener el refrigerante del torno?,semántica,TC-420,0,NaN,correcto,correcto,1.0,1.0
7,"La inyectora marca E-17, ¿qué hago?",código,IP-250,0,NaN,correcto,correcto,1.0,1.0
8,¿A qué temperatura se inyecta el ABS?,código,IP-250,1,0.9684,todos,correcto,1.0,1.0
9,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?,semántica,CB-1200,1,0.9508,correcto,correcto,2.0,1.0


**Análisis.**

- **Manual:** con lo aprendido, el sistema eligió el manual correcto en 13 de 14 casos (antes 10) y no eligió ninguno equivocado. Los tres que pasaron de «todos» a «correcto» son preguntas que no nombran la máquina (el husillo, el ABS y la banda que resbala). El único «todos» que queda es la pregunta fuera de dominio, que no se parece a ninguna consulta validada.
- **Recuperación:** el Hit@4 no cambió (0,923) y el MRR subió de 0,885 a 0,923. En «¿cada cuánto se engrasan los rodamientos…?», el fragmento correcto pasó del puesto 2 al 1, porque la respuesta validada lo había citado.
- **Donde no alcanzó:** en «el torno se queda sin fuerza…» se encontró la consulta validada (0,899), pero su respuesta había citado la sección de seguridad y el procedimiento ante AL-205, no la tabla de alarmas que dice «presión hidráulica baja», que es lo que pide el criterio. La memoria refuerza lo que la respuesta validada usó, no necesariamente el fragmento que eligió el evaluador. La respuesta final igual fue correcta (sección 11.5).
- **Sin efectos colaterales:** los 8 casos sin consultas parecidas quedaron exactamente igual, porque por debajo del umbral la memoria no interviene.

La prueba tiene un límite claro: las consultas validadas repiten a los casos de prueba con otras palabras. Mide lo que pasa en una planta cuando se repite una consulta, que es el caso de uso, pero no que el sistema generalice a temas nuevos.

### 11.4 Una consulta nueva con lo aprendido

Una pregunta que no está en ninguno de los dos conjuntos: no nombra la máquina, así que sin memoria el sistema buscaría en todos los manuales. Se muestra también el bloque de consultas validadas que recibe el LLM.

In [34]:
def preguntar_con_memoria(pregunta):
    salida = cadena_rag_mem.invoke({"input": pregunta, "historial": []})
    plan = salida["plan"]
    print(f"Manual: {plan['manual'] or 'todos'} ({plan['origen'] or 'sin filtro'})")
    print("\nConsultas validadas que recibe el LLM:\n" + formatear_validadas(salida["recuerdos"]))
    print("\nRespuesta:\n" + salida["respuesta"])
    print("\nFuentes:")
    for i, d in enumerate(salida["fuentes"], start=1):
        m = d.metadata
        print(f"  [{i}] {m['modelo']} - {m['seccion']}  (memoria: {m.get('score_memoria', '-')})")


preguntar_con_memoria("¿Cuál es la temperatura para inyectar ABS?")

Manual: IP-250 (aprendido)

Consultas validadas que recibe el LLM:
Pregunta: ¿Qué temperatura de inyección lleva el ABS?
Respuesta validada: Para procesar ABS en la inyectora PLASTIMAQ IP-250, la temperatura del barril debe estar entre **210 °C y 240 °C**.

Recuerde que, según el manual, el ABS debe secarse previamente durante **2 horas a 80 °C** antes de ser procesado.

Respuesta:
Para procesar ABS en la inyectora PLASTIMAQ IP-250, la temperatura del barril debe estar entre **210 °C y 240 °C**, y la temperatura del molde entre **50 °C y 70 °C** [1].

Recuerde que, según el manual, el ABS debe secarse previamente durante **2 horas a 80 °C** antes de ser procesado [1].

Fuentes:
  [1] IP-250 - 3. Operación > 3.1 Temperaturas de procesamiento recomendadas  (memoria: 0.9676)
  [2] IP-250 - 3. Operación > 3.2 Purga y cambio de material  (memoria: -)
  [3] IP-250 - 1. Descripción general > 1.1 Datos técnicos  (memoria: -)
  [4] IP-250 - 5. Códigos de error del control PQ-Touch  (memoria: -)

**Análisis.** La pregunta no nombra la inyectora, pero el sistema eligió la IP-250 por lo aprendido (origen «aprendido»). El fragmento de temperaturas llegó primero, con el puntaje de la memoria, y el LLM recibió la respuesta validada sin sus citas. La respuesta nueva no la copió: tomó los datos del contexto, los citó con [1] y agregó la temperatura del molde, que la validada no tenía.

### 11.5 Respuestas con memoria

Se corre la cadena completa, ahora con memoria, en los casos de prueba que encontraron una consulta validada parecida, para comprobar que las respuestas siguen siendo correctas.

In [35]:
casos_con_memoria = [caso for caso, n in zip(CASOS, df_mem["recuerdos"]) if n]
df_resp_mem = evaluar_respuestas(cadena_rag_mem, pausa=4, casos=casos_con_memoria)
print(f"Respuestas correctas con memoria: {df_resp_mem['correcta'].sum()} de {len(df_resp_mem)}")
df_resp_mem[["tipo", "pregunta", "recuerdos", "correcta", "respuesta"]]

Respuestas correctas con memoria: 6 de 6


,tipo,pregunta,recuerdos,correcta,respuesta
0,semántica,"El torno se queda sin fuerza para sujetar la pieza en el plato, parece falta de aceite en el sis...",1,True,"Para investigar la falta de fuerza en el plato del torno TC-420, primero tené en cuenta las sigu..."
1,semántica,¿Cómo caliento el husillo si la máquina estuvo parada todo el fin de semana?,1,True,"Como la máquina estuvo detenida más de 48 horas (todo el fin de semana), debes ejecutar el progr..."
2,código,¿A qué temperatura se inyecta el ABS?,1,True,"Para procesar ABS en la inyectora PLASTIMAQ IP-250, la temperatura del barril debe estar entre *..."
3,semántica,¿Cada cuánto se engrasan los rodamientos de la cinta transportadora?,1,True,Los rodamientos de los tambores de la cinta transportadora CB-1200 deben reengrasarse cada **500...
4,semántica,"La banda resbala sobre el tambor que la mueve, ¿qué puede ser?",1,True,"Para solucionar el patinamiento de la banda en el tambor motriz de la TRANSBELT CB-1200, debés c..."
5,semántica,"Un compañero se fue a su casa y dejó su candado puesto en la máquina, ¿quién lo puede sacar?",1,True,"Si un operario dejó su candado colocado y no se encuentra en la planta, el procedimiento estable..."


**Análisis.** Los 6 casos de prueba que usaron una consulta validada siguieron respondiendo bien (6 de 6). Pasarle al LLM la respuesta validada como ejemplo no hizo que inventara ni que mezclara datos.

In [36]:
memoria_eval.vs.delete_collection()  # la memoria de la prueba era temporal: la del server no se tocó

## 12. Aplicación web

Además del notebook, el sistema tiene una interfaz web para usarlo como chat: un server con FastAPI (`main.py`) que usa este mismo código (carpeta `asistente/`) y un client en React.

- **Consultas:** chat con respuestas en streaming. Cada respuesta muestra el manual elegido y por qué, si usó consultas validadas parecidas, los fragmentos consultados y los botones 👍 / 👎.
- **Manuales:** lista, visor y carga de manuales nuevos.
- **Aprendizaje:** las consultas que el sistema aprendió, con la opción de olvidar cada una.

La conversación vive en el navegador y no se guarda en el server: el client manda el historial en cada consulta y «Nueva» lo descarta. Lo validado con 👍 se guarda en la colección `memoria` de Chroma (carpeta `chroma_db/`), así que sigue disponible después de reiniciar el server.

| Endpoint | Uso |
|---|---|
| `POST /api/consulta` | Pregunta + historial; responde en streaming: plan → memoria → fuentes → texto |
| `GET /api/aprendizaje` | Consultas aprendidas |
| `POST /api/aprendizaje` | 👍: aprende la consulta (texto buscado, respuesta y fragmentos en el orden de las citas) |
| `DELETE /api/aprendizaje/{id}` | 👎 u «Olvidar»: la quita de la memoria |

Para levantarla:

```
# terminal 1, carpeta server
.venv\Scripts\python -m uvicorn main:app --port 8000

# terminal 2, carpeta client
npm run dev
```

y abrir http://localhost:5173.

## 13. Decisiones de diseño

| Decisión | Alternativa | Por qué |
|---|---|---|
| RAG | Agente con herramientas | La tarea es buscar y explicar información de manuales; con RAG alcanza una llamada al LLM por pregunta |
| Manuales de equipos ficticios | Manuales reales | Así sabemos que las respuestas salen de los documentos y no de lo que el modelo ya sabía |
| Dividir por secciones y agregar un encabezado a cada chunk | Cortar solo por tamaño | Evita chunks que no dicen de qué máquina hablan (sección 4) |
| Búsqueda híbrida con RRF | Solo embeddings | Los códigos de alarma se recuperan mejor con BM25 (secciones 7 y 10) |
| Decidir el manual con los alias, sin LLM | Preguntarle al LLM, o buscar siempre en todos | No mezcla máquinas y no gasta una llamada extra |
| Seguimiento sumando la pregunta anterior | Reformular la pregunta con el LLM | Resuelve los casos probados sin otra llamada al modelo |
| Chroma | FAISS | Guarda el índice en disco y permite filtrar por metadata (el manual) |
| `gemini-3.1-flash-lite` con temperatura 0.1 | Modelos más grandes | Es económico, entra en el plan gratuito y sus respuestas son estables |
| Aprender solo lo validado con 👍 | Aprender todo lo conversado | El sistema no aprende ni repite respuestas equivocadas |
| No guardar las conversaciones, solo lo aprendido | Guardar el historial completo | Menos datos guardados; para aprender alcanza con la pregunta, la respuesta y los fragmentos citados |
| Memoria con embeddings y umbral 0,88 | Buscar la pregunta exacta | Reconoce la misma consulta dicha con otras palabras (sección 11.1) |
| Usar lo aprendido para el manual, en RRF y como ejemplo en el prompt | Reentrenar el modelo (fine-tuning) | Se aplica desde la consulta siguiente, no necesita reentrenar y se puede deshacer |
| Guardar las preguntas con embedding de consulta (`RETRIEVAL_QUERY`) | Embedding de documento (el de los chunks) | En la memoria se compara pregunta con pregunta |

## 14. Dificultades y cómo las resolvimos

| Dificultad | Causa | Solución |
|---|---|---|
| Se recuperaban fragmentos de otra máquina | Al cortar solo por tamaño, muchos chunks no nombran el equipo | Dividir por secciones, encabezado en cada chunk y filtro por manual |
| Malos resultados al buscar por código de alarma | Los embeddings no distinguen bien códigos como AL-205 y AL-250 | Agregar BM25 con un tokenizador que conserva los códigos |
| BM25 no encuentra "engrasan" cuando el manual dice "reengrasar" | BM25 compara palabras exactas | La búsqueda vectorial cubre esos casos y RRF combina las dos |
| Las preguntas de seguimiento no encontraban nada | "¿Y cuántas veces puedo rearmarlo?" no tiene palabras útiles para buscar | Sumar la pregunta anterior y mantener el manual de la conversación |
| Error 429 (cuota agotada) | El plan gratuito de `gemini-3.6-flash` permitía solo 20 pedidos por día | Cambiar a un modelo *lite* y usar una sola llamada por pregunta |
| El primer modelo que elegimos (`gemini-2.5-flash-lite`) devolvía error 404 | Google lo retiró para cuentas nuevas | Pasamos a `gemini-3.1-flash-lite` |
| Respuestas con fórmulas en LaTeX que la interfaz mostraba mal | El modelo escribe las cuentas así | Regla en el prompt: "No uses LaTeX" |
| Un PDF que no era un manual aparecía como fuente | Se había copiado material de clase a la carpeta de manuales | Se sacó del índice y ahora solo se indexan los manuales `.md` |
| Dos preguntas distintas sobre el aceite del compresor daban 0,867 de similitud | Misma máquina y mismo tema | Umbral 0,88, elegido midiendo paráfrasis y no paráfrasis (sección 11.1) |
| Las citas [n] de una respuesta validada no sirven en una consulta nueva | La numeración del contexto cambia en cada consulta | Se aprenden los ids de los fragmentos citados y se quitan las citas antes de pasar la respuesta al prompt |

## 15. Resultados y conclusiones

**Resultados**

| Qué se midió | Resultado |
|---|---|
| Chunks que no dicen de qué máquina son | 0 de 54 (contra 31 de 42 cortando solo por tamaño) |
| Recuperación, Hit@4 / MRR | vectorial 1,000 / 0,923 · BM25 0,846 / 0,808 · híbrida 0,923 / 0,885 |
| Detección del manual | 10 correctos, 4 "todos" y 0 incorrectos |
| Respuestas correctas | 14 de 14 |
| Llamadas al LLM por pregunta | 1 |
| Aprendizaje (7 consultas validadas) | manual correcto 10 → 13 de 14 (0 incorrectos) · MRR híbrida 0,885 → 0,923 · 6 de 6 respuestas correctas con memoria |

**Conclusiones**

- Con RAG, un modelo general responde con datos específicos de manuales que nunca vio y cita de dónde los sacó. Como los equipos son ficticios, sabemos que la información viene de los documentos.
- La forma de dividir los manuales importa tanto como el buscador: el encabezado con el equipo en cada chunk evitó que se mezclaran datos de máquinas distintas.
- Los embeddings resolvieron bien las preguntas semánticas y BM25 las de códigos. En nuestro set la búsqueda vectorial sola ya alcanzaba, pero la prueba de la sección 5 muestra que confunde códigos parecidos, así que la híbrida es la opción más segura.
- Decidir el manual con reglas simples, sin el LLM, funcionó bien (ningún error) y deja una sola llamada al modelo por pregunta.
- El sistema aprende de las conversaciones sin guardarlas y sin reentrenar el modelo: con 7 consultas validadas eligió bien el manual en 13 de 14 casos (antes 10), sin errores, y el umbral medido evitó reutilizar una consulta distinta sobre el mismo tema.

**Trabajo futuro**

- Probar con manuales reales en PDF, que necesitan extraer tablas y, a veces, OCR.
- Ampliar el set de pruebas, sobre todo con códigos de alarma parecidos entre sí.
- Agregar un re-ranking de los fragmentos recuperados antes de enviarlos al modelo.
- Aprender también de las correcciones: que con 👎 el técnico pueda indicar cuál era el fragmento correcto.
- Probar el aprendizaje con más consultas reales y con validaciones de distintos técnicos.